In [9]:
import os, sys
sys.path.insert(0, os.path.abspath("../.."))   # distributed_framework/

In [12]:
import math
import random
import numpy as np
from typing import List, Optional, Tuple, Union
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from torch import nn
import os
from dataclasses import dataclass, field

from loguru import logger
from torch import nn
from typing import Literal
from model import DeepSeekV3Model
from model_args import DeepSeekV3ModelArgs, MoEArgs
from torchfeather.config.default_configs import get_torchfeather_1b_model_args

## Para Check

In [1]:
model_args = get_torchfeather_1b_model_args()
model = DeepSeekV3Model(model_args)

model_args.get_params_and_flops(model, model_args.max_seq_len)

NameError: name 'get_torchfeather_1b_model_args' is not defined

## RoPE

In [90]:
torch.manual_seed(123)
B, H, S, D = 2, 8, 100, 256 
k = torch.randn((B, S, H, D))
q = torch.randn((B, S, H, D))

In [174]:
def method_1(q):
    freqs = (1 / 10000) ** (torch.arange(0, D, 2).float() / D)  # (D/2,)
    t = torch.arange(S).float()                                 # (S,)
    freqs = torch.outer(t, freqs)                               # (S, D/2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)      # (S, D/2) complex

    dtype = q.dtype
    # (B, S, H, D) -> (B, S, H, D/2, 2) -> (B, S, H, D/2) complex
    x = torch.view_as_complex(q.float().reshape(*q.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, S, 1, -1)                     # (1, S, 1, D/2)
    y = torch.view_as_real(x * freqs_cis).flatten(3)            # (B, S, H, D)
    return y.to(dtype)

In [189]:
q_embed_method_1 = method_1(q)

In [190]:
q_embed_method_1.shape

torch.Size([2, 100, 8, 256])

In [201]:
# method 2: use [cos -sin/n sin cos] matrix with element-wise order trick
def rotate_adjacent(x: torch.Tensor) -> torch.Tensor:
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)

def method_2(q):
    inv_freq = (1 / 10000) ** (torch.arange(0, D, 2).float() / D) # (D/2,)
    t = torch.arange(S).float()                                   # (S,)
    freqs = torch.outer(t, inv_freq)                              # (S, D/2)
    
    # Interleave to match adjacent pairs: [f0, f0, f1, f1, ...]
    freqs = torch.repeat_interleave(freqs, 2, dim=-1)             # (S, D)
    cos = freqs.cos().view(1, S, 1, D)
    sin = freqs.sin().view(1, S, 1, D)
    
    return q * cos + rotate_adjacent(q) * sin


In [202]:
q_embed_method_2 = method_2(q)
q_embed_method_2.shape

torch.Size([2, 100, 8, 256])

In [203]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def method_3(q):
    inv_freq = 1.0 / (10000 ** (torch.arange(0, D, 2).float() / D)) # (D/2,)
    t = torch.arange(S).float()                                     # (S,)
    freqs = torch.outer(t, inv_freq)                                # (S, D/2)
    
    # Concatenate halves: [f0, f1, ..., f0, f1, ...]
    freqs = torch.cat((freqs, freqs), dim=-1)                       # (S, D)
    cos = freqs.cos().view(1, S, 1, D)
    sin = freqs.sin().view(1, S, 1, D)                        
    
    return q * cos + rotate_half(q) * sin

In [204]:
q_embed_method_3 = method_3(q)
q_embed_method_3.shape

torch.Size([2, 100, 8, 256])

In [ ]:
print("Method 1 == Method 2:", torch.allclose(q_embed_method_1, q_embed_method_2, atol=1e-5))  # True
print("Method 2 == Method 3 (direct):", torch.allclose(q_embed_method_2, q_embed_method_3, atol=1e-5))  # False (different dimension pairing)

# Analysis: Method 1 & Method 2 produce the exact same tensor element-by-element, while Method 3 produces a permuted tensor due to how the dimension pairs are grouped:
# Method 1 & 2 (Pairwise / Adjacent Pairing): Group adjacent indices (0, 1), (2, 3), (4, 5) into 2D rotation planes.
# Method 3 (Split-Half Pairing): Groups the first half with the second half (0, D/2), (1, D/2 + 1), (2, D/2 + 2).

Method 1 == Method 2: True
Method 2 == Method 3 (direct): False
